In [5]:
import io
import os
import time

import Levenshtein
from PIL import Image, ImageEnhance
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from msrest.authentication import CognitiveServicesCredentials
from spellchecker import SpellChecker

'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

# img = open("data/images/test1.png", "rb")
img = open("data/images/test2.jpeg", "rb")
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)
# print(read_response.as_dict())

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Print the detected text, line by line
result = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            # print(line.text)
            result.append(line.text)


recognized_text = " ".join(result)
recognized_words = recognized_text.split()
print(recognized_text)
ground_truth_text = "Succes în rezolvarea tEMELOR la LABORAtoarele de Inteligență Aritificială!"
ground_truth_words = ground_truth_text.split()

Lucces in resolvarea TEMELOR la LABORA toarele de Inteligenta Artificialà!


In [6]:
def cerinta1a():
    def CharacterErrorRate():
        distance_char = Levenshtein.distance(recognized_text,
                                             ground_truth_text)  # Calculate the minimum edit distance between the recognized text and the ground truth text
        CER = distance_char / len(ground_truth_text)
        print("Character Error Rate: ", CER)

    def WordErrorRate():
        word_distance = sum(Levenshtein.distance(w1, w2) for w1, w2 in zip(recognized_words, ground_truth_words))
        WER = word_distance / sum(len(w) for w in ground_truth_words)
        print("Word Error Rate: ", WER)

    CharacterErrorRate()
    WordErrorRate()

In [7]:
cerinta1a()

Character Error Rate:  0.12162162162162163
Word Error Rate:  0.5757575757575758


In [8]:
def cerinta1b():
    # CER
    # Levenshtein distance
    cer = Levenshtein.distance(recognized_text, ground_truth_text) / len(ground_truth_text)
    print("Levenshtein Character Error Rate: ", cer)
    # Jaro-Winkler
    jw = Levenshtein.jaro_winkler(recognized_text, ground_truth_text) # The difference between Levenshtein distance is that Jaro-Winkler adds a bonus for the most common prefix between the two strings
    print("Jaro-Winkler Character Error Rate: ", jw)
    # Longest Common Subsequence
    def longest_common_subsequence(s1, s2): # How much from the recognized text is present in the ground truth text, regardless of the order of the characters
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[m][n]
    lcs_error = 1 - longest_common_subsequence(recognized_text, ground_truth_text) / len(ground_truth_text)
    print("LCS Character Error Rate: ", lcs_error)

    # WER
    # Exact match ratio
    wrong_words = sum(1 for w1, w2 in zip(recognized_text.split(), ground_truth_text.split()) if w1 != w2)
    wer = wrong_words / len(ground_truth_text.split())
    print("Word Error Rate: ", wer)
    # Levenshtein distance
    word_distance = sum(Levenshtein.distance(w1, w2) for w1, w2 in zip(recognized_words, ground_truth_words))
    wer = word_distance / sum(len(w) for w in ground_truth_words)
    print("Levenshtein Word Error Rate: ", wer)

In [9]:
cerinta1b()

Levenshtein Character Error Rate:  0.12162162162162163
Jaro-Winkler Character Error Rate:  0.8672851956434046
LCS Character Error Rate:  0.10810810810810811
Word Error Rate:  0.8888888888888888
Levenshtein Word Error Rate:  0.5757575757575758
